In [1]:
import sys  
sys.path.insert(1, '../src')

from pathlib import Path
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from skimage.morphology import local_maxima

from UNet_class_and_functions import *

from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
PROJECT_ROOT = Path.cwd().parent

LOGS_PATH = PROJECT_ROOT / 'logs' / 'wing_extraction_log.csv'
CROP_ROOT = PROJECT_ROOT / 'data' / 'images' / 'wing_crops'
PREDICT_ROOT = PROJECT_ROOT / 'out' / 'predict'
MODEL_DIR = PROJECT_ROOT / 'data' / 'models' / 'best_model'

METADATAS_OUT = PROJECT_ROOT / 'data' / 'annotations' / 'metadatas.csv'
LANDMARKS_OUT = PROJECT_ROOT / 'data' / 'annotations' / 'landmarks.tps'

print(f'Racine du projet : {PROJECT_ROOT}')
print(f'Crop root : {CROP_ROOT}')

Racine du projet : c:\Users\Jules\Documents\IEES\idmybee
Crop root : c:\Users\Jules\Documents\IEES\idmybee\data\images\wing_crops


In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model_name = "UNet_150_epoch_lr=0.001_seed=58_func=pow_param=30"
model_load = torch.load(MODEL_DIR / (model_name + '.pth'),
                        weights_only=False, map_location=device)
model_load.to(device)
model_load.eval()

wing_crops = pd.read_csv(LOGS_PATH)

In [8]:
def get_species(row):
    try:
        sp = row["image"].split('\\')[0].split('_')[1]
    except:
        sp = row["image"].split('\\')[0]
    return sp

def get_role(row):
    return row["image"].split('\\')[1]

def get_device(row):
    return row["name"].split('_')[-1]

valids = wing_crops.loc[wing_crops.status == 'OK']
valids["espece"] = valids.apply(get_species, axis=1)
valids["role"] = valids.apply(get_role, axis=1)
valids["device"] = valids.apply(get_device, axis=1)

valids = valids.drop(valids[valids.espece == "Vrac"].index).reset_index()
valids[["name", "espece", "role", "device"]].to_csv(METADATAS_OUT, index_label="id")

In [5]:
crops = []
crops_t = []
out_crops = []
annotations = []

records = []

for i, row in tqdm(valids.iterrows(), total=valids.shape[0]):
    path = CROP_ROOT / row.image

    crop = cv2.imread(path)
    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    crops.append(crop)

    crop_t = torch.Tensor(crop.transpose(2,0,1)/255).unsqueeze(0)
    crops_t.append(crop_t)

    crop_t = crop_t.to(device)
    output = model_load(crop_t).cpu().squeeze(0).detach().numpy().transpose(1,2,0)
    out_crops.append(output)

    output = output.squeeze(axis = 2)
    maximas = np.argwhere(local_maxima(output) == True)
    scores = np.array([output[pt[0],pt[1]] for pt in maximas])
    sorter = np.argsort(scores)
    maximas = maximas[sorter,:]
    maximas = maximas[-18:,:]
    annotations.append(maximas)

    records.append({
        "landmarks": maximas,
        "image_path": path,
        "specimen_id": i,
    })

  0%|          | 0/2732 [00:00<?, ?it/s]

In [6]:
with open(LANDMARKS_OUT, "w", encoding="utf-8") as f:
    for rec in records:
        lm = rec["landmarks"]
        f.write(f"LM={len(lm)}\n")
        for y, x in lm:
            f.write(f"{x:.4f} {y:.4f}\n")
        f.write(f"IMAGE={rec['image_path']}\n")
        f.write(f"ID={rec['specimen_id']}\n")